# 02 — Data Cleaning & Feature Engineering

Phase A (null audit / cleaning), Phase B (`player_key` identity resolution),
and Phase C (eligible population + wide-format reshape + train/test split)
of the forecasting pipeline.

Input: `data/processed/merged.csv`
Outputs: `data/processed/cleaned_merged.csv`, `data/processed/player_seasons_merged.csv`,
`data/processed/phase_c_eligible_players.csv`

Modeling picks up in `03_forecasting.ipynb`.

## Phase A — Null audit & cleaning

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/merged.csv')
print(df.shape)
print(df.dtypes)

(9626, 33)
league                                str
season                              int64
team                                  str
player                                str
nation                                str
pos                                   str
age                               float64
matches                           float64
starts                            float64
minutes                           float64
goals                             float64
assists                           float64
goal_contributions                float64
goals_excl_pk                     float64
pk_scored                         float64
pk_attempted                      float64
yellow_cards                      float64
red_cards                         float64
goals_p90                         float64
assists_p90                       float64
goal_contributions_p90            float64
goals_excl_pk_p90                 float64
goal_contributions_excl_pk_p90    float64
shots                  

In [2]:
df.isnull().sum()

league                               0
season                               0
team                                 0
player                               0
nation                               1
pos                                  0
age                                  0
matches                              0
starts                               0
minutes                              0
goals                                0
assists                              0
goal_contributions                   0
goals_excl_pk                        0
pk_scored                            0
pk_attempted                         0
yellow_cards                         0
red_cards                            0
goals_p90                            0
assists_p90                          0
goal_contributions_p90               0
goals_excl_pk_p90                    0
goal_contributions_excl_pk_p90       0
shots                               84
shots_on_target                     84
shot_accuracy            

In [3]:
# Which teams have null league values?
teams_with_null_league = df[df['league'].isnull()]['team'].unique()
print(teams_with_null_league)

# Do ANY of these teams appear with a non-null league elsewhere? (would contradict your theory)
for team in teams_with_null_league:
    leagues_for_team = df[df['team'] == team]['league'].dropna().unique()
    print(team, '->', leagues_for_team)

<StringArray>
[]
Length: 0, dtype: str


In [4]:
df['league'] = df['league'].fillna('GER-Bundesliga')

# Verify
print(df['league'].isnull().sum())
print(df['league'].unique())

0
<StringArray>
['ENG-Premier League',        'ESP-La Liga',        'FRA-Ligue 1',
        'ITA-Serie A',     'GER-Bundesliga']
Length: 5, dtype: str


In [5]:
df.loc[df['player'] == 'Jeremy Ngakia', 'nation'] = 'COD'

# Verify
print(df['nation'].isnull().sum())

0


In [6]:
# Which season(s) do these null rows belong to?
null_shots_rows = df[df['shots'].isnull()]
print(null_shots_rows['season'].value_counts())

# Do the other three columns have nulls in exactly the same rows, or different ones?
print((df['shots'].isnull() == df['shots_on_target'].isnull()).all())
print((df['shots'].isnull() == df['shots_p90'].isnull()).all())
print((df['shots'].isnull() == df['shots_on_target_p90'].isnull()).all())

season
1920    84
Name: count, dtype: int64
True
True
True


In [7]:
# Are these 84 players low-minute players, or did they play normal minutes?
print(df[df['shots'].isnull()]['minutes'].describe())

count      84.000000
mean     1795.273810
std       618.639634
min       917.000000
25%      1317.250000
50%      1629.000000
75%      2221.500000
max      3420.000000
Name: minutes, dtype: float64


In [8]:
players_missing_1920_shots = df[df['shots'].isnull()]['player'].unique()
print(len(players_missing_1920_shots))

# Do these same players have valid shot data in OTHER seasons?
other_seasons_data = df[(df['player'].isin(players_missing_1920_shots)) & (df['season'] != 1920)]
print(other_seasons_data['shots'].isnull().sum(), 'out of', len(other_seasons_data), 'rows')

84
0 out of 231 rows


In [9]:
df = df[df['shots'].notnull()].reset_index(drop=True)

# Verify
print(df.shape)
print(df[['shots', 'shots_on_target', 'shots_p90', 'shots_on_target_p90']].isnull().sum())

(9542, 33)
shots                  0
shots_on_target        0
shots_p90              0
shots_on_target_p90    0
dtype: int64


In [10]:
df.isnull().sum()

league                               0
season                               0
team                                 0
player                               0
nation                               0
pos                                  0
age                                  0
matches                              0
starts                               0
minutes                              0
goals                                0
assists                              0
goal_contributions                   0
goals_excl_pk                        0
pk_scored                            0
pk_attempted                         0
yellow_cards                         0
red_cards                            0
goals_p90                            0
assists_p90                          0
goal_contributions_p90               0
goals_excl_pk_p90                    0
goal_contributions_excl_pk_p90       0
shots                                0
shots_on_target                      0
shot_accuracy            

In [11]:
# Hypothesis: shot_accuracy / goals_per_shot are null exactly when shots == 0
print((df['shot_accuracy'].isnull() == (df['shots'] == 0)).all())
print((df['goals_per_shot'].isnull() == (df['shots'] == 0)).all())

# Hypothesis: goals_per_shot_on_target is null exactly when shots_on_target == 0
print((df['goals_per_shot_on_target'].isnull() == (df['shots_on_target'] == 0)).all())

True
True
True


In [12]:
# For players with shots == 0, are goals also always 0?
print(df[df['shots'] == 0]['goals'].describe())

count    695.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: goals, dtype: float64


In [13]:
df['shot_accuracy'] = df['shot_accuracy'].fillna(0)
df['goals_per_shot'] = df['goals_per_shot'].fillna(0)
df['goals_per_shot_on_target'] = df['goals_per_shot_on_target'].fillna(0)

# Verify
print(df.isnull().sum().sum())

0


In [14]:
df['season'] = df['season'].astype(str)

# Verify
print(df['season'].dtype)
print(df['season'].unique())

str
<StringArray>
['1920', '2021', '2122', '2223', '2324', '2425']
Length: 6, dtype: str


In [15]:
print(df['pos'].value_counts())

pos
MF       2751
DF       2707
FW        937
MF,FW     840
GK        709
DF,MF     603
FW,MF     538
MF,DF     451
DF,FW       6
Name: count, dtype: int64


In [16]:
def standardize_pos(pos):
    parts = pos.split(',')
    parts_sorted = sorted(parts)
    return ','.join(parts_sorted)

df['pos'] = df['pos'].apply(standardize_pos)

# Verify
print(df['pos'].value_counts())

pos
MF       2751
DF       2707
FW,MF    1378
DF,MF    1054
FW        937
GK        709
DF,FW       6
Name: count, dtype: int64


In [17]:
# 1. Exact duplicate rows?
print("Exact duplicate rows:", df.duplicated().sum())

# 2. Impossible logical relationships
print("starts > matches:", (df['starts'] > df['matches']).sum())
print("shots_on_target > shots:", (df['shots_on_target'] > df['shots']).sum())
print("pk_scored > pk_attempted:", (df['pk_scored'] > df['pk_attempted']).sum())
print("goals < pk_scored:", (df['goals'] < df['pk_scored']).sum())

# 3. Does goals_excl_pk actually equal goals - pk_scored?
mismatch = (df['goals_excl_pk'] != (df['goals'] - df['pk_scored'])).sum()
print("goals_excl_pk mismatches:", mismatch)

# 4. Sanity ranges
print(df[['age','minutes','matches','starts']].describe())

Exact duplicate rows: 0
starts > matches: 0
shots_on_target > shots: 0
pk_scored > pk_attempted: 0
goals < pk_scored: 0
goals_excl_pk mismatches: 0


               age      minutes      matches       starts
count  9542.000000  9542.000000  9542.000000  9542.000000
mean     26.070845  1954.630057    27.287571    22.129428
std       4.077132   669.494780     6.608752     7.878833
min      16.000000   900.000000    10.000000     6.000000
25%      23.000000  1376.250000    23.000000    15.000000
50%      26.000000  1898.500000    28.000000    22.000000
75%      29.000000  2495.750000    33.000000    29.000000
max      42.000000  3420.000000    38.000000    38.000000


In [18]:
df.to_csv('../data/processed/cleaned_merged.csv', index=False)

## Phase B — `player_key` identity resolution (name collisions + transfers)

In [19]:
# How many unique player names are there, vs how many rows?
print("Total rows:", len(df))
print("Unique player names:", df['player'].nunique())

# For each player name, how many DIFFERENT nations appear under that name?
# If a name always maps to 1 nation, that's reassuring (no collision by nationality at least).
name_nation_counts = df.groupby('player')['nation'].nunique().sort_values(ascending=False)
print("\nNames with more than 1 nation attached:")
print(name_nation_counts[name_nation_counts > 1])

Total rows: 9542
Unique player names: 3588

Names with more than 1 nation attached:
player
João Pedro          2
Adama Traoré        2
Fernando            2
Luis Suárez         2
Nicolás González    2
Name: nation, dtype: int64


In [20]:
# Look at the actual rows for each flagged name — nation, team, season side by side
flagged_names = ['Nicolás González', 'Luis Suárez', 'Fernando', 'Adama Traoré', 'João Pedro']

for name in flagged_names:
    print(f"\n=== {name} ===")
    print(df[df['player'] == name][['season', 'team', 'nation', 'league', 'minutes']].sort_values('season'))


=== Nicolás González ===
     season        team nation          league  minutes
8357   2021   Stuttgart    ARG  GER-Bundesliga    934.0
2669   2122   Barcelona    ESP     ESP-La Liga   1114.0
6585   2122  Fiorentina    ARG     ITA-Serie A   2357.0
3260   2223    Valencia    ESP     ESP-La Liga   1405.0
6926   2223  Fiorentina    ARG     ITA-Serie A   1354.0
7262   2324  Fiorentina    ARG     ITA-Serie A   1913.0
7682   2425    Juventus    ARG     ITA-Serie A   1750.0

=== Luis Suárez ===
     season             team nation       league  minutes
1997   1920        Barcelona    URU  ESP-La Liga   1998.0
2307   2021  Atlético Madrid    URU  ESP-La Liga   2508.0
2423   2021          Granada    COL  ESP-La Liga   1666.0
2648   2122  Atlético Madrid    URU  ESP-La Liga   1835.0
2768   2122          Granada    COL  ESP-La Liga   2844.0
2959   2223          Almería    COL  ESP-La Liga   1490.0
3331   2324          Almería    COL  ESP-La Liga    917.0

=== Fernando ===
     season     team na

In [21]:
# Build the player_key now (name + nation)
df['player_key'] = df['player'] + "_" + df['nation']

# Find player_key + season combos that show up more than once
dupe_counts = df.groupby(['player_key', 'season']).size()
dupe_combos = dupe_counts[dupe_counts > 1]

print("Number of player_key+season combos with duplicates:", len(dupe_combos))
print(dupe_combos)

Number of player_key+season combos with duplicates: 66
player_key            season
Adrien Thomasson_FRA  2223      2
Amine Gouiri_ALG      2425      2
Anthony Rouault_FRA   2425      2
Antonio Candela_ITA   2425      2
Antonio Sanabria_PAR  2021      2
                               ..
Vitinha_POR           2425      2
Weston McKennie_USA   2223      2
Wout Weghorst_NED     2122      2
Yann Sommer_SUI       2223      2
Álex Moreno_ESP       2223      2
Length: 66, dtype: int64


In [22]:
# Get the list of player_key+season combos that are duplicated
dupe_keys = dupe_combos.index  # this is a list of (player_key, season) pairs

# Pull all rows belonging to those combos, sorted for easy reading
flagged_rows = df[df.set_index(['player_key', 'season']).index.isin(dupe_keys)]
flagged_rows_sorted = flagged_rows.sort_values(['player_key', 'season', 'team'])

print(flagged_rows_sorted[['player_key', 'season', 'team', 'minutes', 'league']].to_string())

                         player_key season                 team  minutes              league
5009           Adrien Thomasson_FRA   2223                 Lens   1278.0         FRA-Ligue 1
5202           Adrien Thomasson_FRA   2223           Strasbourg    937.0         FRA-Ligue 1
5664               Amine Gouiri_ALG   2425            Marseille   1048.0         FRA-Ligue 1
5775               Amine Gouiri_ALG   2425               Rennes   1037.0         FRA-Ligue 1
5776            Anthony Rouault_FRA   2425               Rennes    972.0         FRA-Ligue 1
9481            Anthony Rouault_FRA   2425            Stuttgart   1212.0      GER-Bundesliga
3948            Antonio Candela_ITA   2425           Valladolid    906.0         ESP-La Liga
7842            Antonio Candela_ITA   2425              Venezia   1008.0         ITA-Serie A
2485           Antonio Sanabria_PAR   2021           Real Betis    932.0         ESP-La Liga
6477           Antonio Sanabria_PAR   2021               Torino    978

In [23]:
# Rule 1: minutes-cap test. Summed minutes per player_key+season exceeding 3420
# is a physical impossibility for one person in one season (38 matchdays x 90 min max).
combo_minutes = flagged_rows.groupby(['player_key', 'season'])['minutes'].sum()
cap_flagged = combo_minutes[combo_minutes > 3420]

print("Flagged by minutes-cap rule (proven collisions):")
print(cap_flagged)

Flagged by minutes-cap rule (proven collisions):
player_key       season
Marcelo_BRA      2021      3957.0
Nacho_ESP        2021      3813.0
Raúl García_ESP  1920      3881.0
Rodri_ESP        2223      4512.0
                 2324      3982.0
Name: minutes, dtype: float64


In [24]:
# Rule 2: repeat-season test. A player_key showing up as a duplicate in MORE than
# one separate season is suspicious — a real transfer is a one-off event, not a
# recurring pattern. This catches collisions the minutes-cap rule can miss.
repeat_keys = dupe_combos.reset_index()['player_key'].value_counts()
repeat_keys = repeat_keys[repeat_keys > 1]

print("player_keys duplicated across multiple different seasons:")
print(repeat_keys)

player_keys duplicated across multiple different seasons:
player_key
Kevin Vogt_GER    2
Marcelo_BRA       2
Rodri_ESP         2
Vitinha_POR       2
Name: count, dtype: int64


In [25]:
# Derive each league's true matchday ceiling directly from the data itself,
# rather than assuming standard numbers that might not hold for every season
# (e.g. Ligue 1's matchday count changed when it dropped to 18 clubs)
league_caps = df.groupby('league')['matches'].max() * 90
print("Empirically-derived per-league minute ceiling:")
print(league_caps)

Empirically-derived per-league minute ceiling:
league
ENG-Premier League    3420.0
ESP-La Liga           3420.0
FRA-Ligue 1           3420.0
GER-Bundesliga        3060.0
ITA-Serie A           3420.0
Name: matches, dtype: float64


In [26]:
def get_cap_for_combo(g):
    leagues_involved = g['league'].unique()
    # If the transfer stayed within one league, use that league's own cap.
    # If it crossed leagues, use the larger of the two caps — the conservative
    # choice, since we'd rather risk missing a rare edge case than wrongly
    # flag a real cross-league transfer as a collision.
    return league_caps[leagues_involved].max()

combo_minutes = flagged_rows.groupby(['player_key', 'season'])['minutes'].sum()
combo_caps = flagged_rows.groupby(['player_key', 'season']).apply(get_cap_for_combo)

cap_flagged = combo_minutes[combo_minutes > combo_caps]
print("Flagged by league-aware minutes-cap rule:")
print(cap_flagged)

Flagged by league-aware minutes-cap rule:


player_key       season
Marcelo_BRA      2021      3957.0
Nacho_ESP        2021      3813.0
Raúl García_ESP  1920      3881.0
Rodri_ESP        2223      4512.0
                 2324      3982.0
Name: minutes, dtype: float64


In [27]:
# The minutes-cap rule mechanically proves these are collisions (physically impossible for one person)
mechanically_proven_collisions = cap_flagged.index.tolist()
print("Mechanically proven (minutes-cap rule):")
print(mechanically_proven_collisions)

# The repeat-season check flags CANDIDATES, not proof — Kevin Vogt and Vitinha needed
# manual inspection of team/league/position to distinguish real transfers from collisions.
# Kevin Vogt (1920, 2324): inspected rows show one continuous DF career across Hoffenheim
#   -> Union Berlin, consistent minutes trajectory. Confirmed ONE real person, safe merge.
# Vitinha (2324, 2425): inspected rows show two disjoint identities — a settled PSG
#   midfielder (Vítor Machado Ferreira) vs a Marseille->Genoa forward (Vítor Manuel
#   Carvalho Oliveira). Confirmed via web search. TWO real people, must escalate.
manually_confirmed_collision_seasons = [('Vitinha_POR', '2324'), ('Vitinha_POR', '2425')]

# Final collision list = mechanically proven + manually confirmed
final_collisions = mechanically_proven_collisions + manually_confirmed_collision_seasons
print("\nFinal collision list (7 combos):")
for c in final_collisions:
    print(c)

Mechanically proven (minutes-cap rule):
[('Marcelo_BRA', '2021'), ('Nacho_ESP', '2021'), ('Raúl García_ESP', '1920'), ('Rodri_ESP', '2223'), ('Rodri_ESP', '2324')]

Final collision list (7 combos):
('Marcelo_BRA', '2021')
('Nacho_ESP', '2021')
('Raúl García_ESP', '1920')
('Rodri_ESP', '2223')
('Rodri_ESP', '2324')
('Vitinha_POR', '2324')
('Vitinha_POR', '2425')


In [28]:
# Escalate every collision by appending team name to split the real people apart.
# Vitinha needs a special rule (PSG vs not) since his split isn't team-exclusive
# the way the others are — see manual inspection notes above.
for (pkey, season) in final_collisions:
    if pkey == 'Vitinha_POR':
        continue  # handled separately below
    mask = (df['player_key'] == pkey) & (df['season'] == season)
    df.loc[mask, 'player_key'] = df.loc[mask, 'player_key'] + '_' + df.loc[mask, 'team']

vitinha_mask = df['player_key'] == 'Vitinha_POR'
df.loc[vitinha_mask, 'player_key'] = df.loc[vitinha_mask].apply(
    lambda row: 'Vitinha_POR_PSG' if row['team'] == 'Paris Saint-Germain' else 'Vitinha_POR_Oliveira',
    axis=1
)

# Verify escalation held
print("Collision check:")
print(df[df['player_key'].str.contains('Marcelo_BRA|Vitinha_POR|Rodri_ESP', regex=True, na=False)]['player_key'].unique())

# Rebuild dupes fresh and confirm count
dupes = df[df.duplicated(subset=['player_key', 'season'], keep=False)].sort_values(['player_key', 'season'])
print(f"\nDuplicate rows: {len(dupes)}")   # expect 118
print(f"Duplicate combos: {dupes.groupby(['player_key','season']).ngroups}")  # expect 59

Collision check:
<StringArray>
[                'Rodri_ESP', 'Rodri_ESP_Manchester City',
               'Marcelo_BRA',   'Marcelo_BRA_Real Madrid',
      'Rodri_ESP_Real Betis',          'Marcelo_BRA_Lyon',
           'Vitinha_POR_PSG',      'Vitinha_POR_Oliveira']
Length: 8, dtype: str



Duplicate rows: 118
Duplicate combos: 59


In [29]:
# Cabrera check — confirmed earlier with Parzaan as a genuine coincidence
# (17 matches x 90 min/match = 1530 at Espanyol; 18 matches x 85 avg min/match = 1530 at Getafe)
print(df[df['player_key'] == 'Leandro Cabrera_URU'][['team', 'season', 'matches', 'starts', 'minutes']])

# Merge function: sum counting stats, recompute all rates fresh from summed counts
sum_cols = ['matches', 'starts', 'minutes', 'goals', 'assists', 'goal_contributions',
            'goals_excl_pk', 'pk_scored', 'pk_attempted', 'yellow_cards', 'red_cards',
            'shots', 'shots_on_target']

def merge_group(g):
    out = {}
    for col in sum_cols:
        out[col] = g[col].sum()
    out['goals_p90'] = out['goals'] / out['minutes'] * 90
    out['assists_p90'] = out['assists'] / out['minutes'] * 90
    out['goal_contributions_p90'] = out['goal_contributions'] / out['minutes'] * 90
    out['goals_excl_pk_p90'] = out['goals_excl_pk'] / out['minutes'] * 90
    out['goal_contributions_excl_pk_p90'] = (out['goals_excl_pk'] + out['assists']) / out['minutes'] * 90
    out['shots_p90'] = out['shots'] / out['minutes'] * 90
    out['shots_on_target_p90'] = out['shots_on_target'] / out['minutes'] * 90
    out['shot_accuracy'] = out['shots_on_target'] / out['shots'] if out['shots'] > 0 else 0
    out['goals_per_shot'] = out['goals'] / out['shots'] if out['shots'] > 0 else 0
    out['goals_per_shot_on_target'] = out['goals'] / out['shots_on_target'] if out['shots_on_target'] > 0 else 0
    dominant_row = g.loc[g['minutes'].idxmax()]
    for col in ['team', 'league', 'pos', 'age', 'player', 'nation']:
        out[col] = dominant_row[col]
    # player_key and season are the groupby keys -- pandas 3.0 no longer
    # includes them as columns inside g, so pull them from the group's
    # own identity (g.name) instead of reading them off a row
    out['player_key'], out['season'] = g.name
    return pd.Series(out)

df_deduped = df[~df.duplicated(subset=['player_key', 'season'], keep=False)].copy()
merged_dupes = dupes.groupby(['player_key', 'season']).apply(merge_group, include_groups=False).reset_index(drop=True)
df_phaseB = pd.concat([df_deduped, merged_dupes], ignore_index=True)

assert df_phaseB.duplicated(subset=['player_key', 'season']).sum() == 0, "Duplicates still remain!"
print(f"\nFinal rows: {len(df_phaseB)}")  # expect 9483

df_phaseB['season'] = df_phaseB['season'].astype(str)
df_phaseB.to_csv('../data/processed/player_seasons_merged.csv', index=False)
print(f"Saved {len(df_phaseB)} rows to player_seasons_merged.csv")

          team season  matches  starts  minutes
2046  Espanyol   1920     17.0    17.0   1530.0
2064    Getafe   1920     18.0    18.0   1530.0
2737  Espanyol   2122     37.0    37.0   3330.0
3078  Espanyol   2223     32.0    31.0   2669.0
3739  Espanyol   2425     33.0    32.0   2869.0



Final rows: 9483


Saved 9483 rows to player_seasons_merged.csv


In [30]:
# Map season labels to a numeric year-index so we can measure the REAL gap between them
season_order = ['1920', '2021', '2122', '2223', '2324', '2425']
season_to_idx = {s: i for i, s in enumerate(season_order)}

def check_age_continuity_fixed(g):
    g = g.sort_values('season').copy()
    g['season_idx'] = g['player_key'].index.map(lambda x: None)  # placeholder, real mapping below
    g['season_idx'] = g['season'].map(season_to_idx)
    g['expected_age_diff'] = g['season_idx'].diff()
    g['actual_age_diff'] = g['age'].diff()
    # Allow some slack (±1) for birthday timing within a season, but the diffs
    # should track each other — actual age gain should roughly match seasons elapsed
    mismatch = (g['actual_age_diff'] - g['expected_age_diff']).abs() > 1
    return mismatch.any()

flagged_fixed = df_phaseB.groupby('player_key').filter(
    lambda g: len(g) > 1 and check_age_continuity_fixed(g)
)
print(f"Player_keys with genuinely suspicious age jumps: {flagged_fixed['player_key'].nunique()}")
print(flagged_fixed[['player_key','season','team','age']].sort_values(['player_key','season']))

Player_keys with genuinely suspicious age jumps: 4
               player_key season           team   age
2013      Diego López_ESP   1920       Espanyol  37.0
2695      Diego López_ESP   2122       Espanyol  39.0
3571      Diego López_ESP   2324       Valencia  21.0
3897      Diego López_ESP   2425       Valencia  22.0
2016       Javi López_ESP   1920       Espanyol  33.0
3274       Javi López_ESP   2324         Alavés  21.0
3867       Javi López_ESP   2425  Real Sociedad  22.0
5783  Ladislav Krejčí_CZE   1920        Bologna  27.0
3734  Ladislav Krejčí_CZE   2425         Girona  25.0
1915      Manu García_ESP   1920         Alavés  33.0
2242      Manu García_ESP   2021         Alavés  34.0
2578      Manu García_ESP   2122         Alavés  23.0


In [31]:
season_order = ['1920', '2021', '2122', '2223', '2324', '2425']
season_to_idx = {s: i for i, s in enumerate(season_order)}

def split_by_age_break(g):
    g = g.sort_values('season').copy()
    g['season_idx'] = g['season'].map(season_to_idx)
    person_id = [0]
    ids = [0]
    for i in range(1, len(g)):
        expected_gap = g['season_idx'].iloc[i] - g['season_idx'].iloc[i-1]
        actual_gap = g['age'].iloc[i] - g['age'].iloc[i-1]
        if abs(actual_gap - expected_gap) > 1:
            person_id[0] += 1
        ids.append(person_id[0])
    g['person_id'] = ids
    return g

flagged_keys = flagged_fixed['player_key'].unique()
for pkey in flagged_keys:
    mask = df_phaseB['player_key'] == pkey
    subset = split_by_age_break(df_phaseB[mask])
    suffix_map = dict(zip(subset.index, subset['person_id'].map(lambda x: chr(65 + x))))
    df_phaseB.loc[subset.index, 'player_key'] = pkey + '_' + subset.index.map(suffix_map)

# Verify the split
print(df_phaseB[df_phaseB['player_key'].str.contains('Diego López|Javi López|Krejčí|Manu García', regex=True, na=False)][['player_key','season','team','age']].sort_values(['player_key','season']))

# Confirm no duplicates were introduced and re-save
assert df_phaseB.duplicated(subset=['player_key', 'season']).sum() == 0
df_phaseB.to_csv('../data/processed/player_seasons_merged.csv', index=False)
print(f"\nRe-saved {len(df_phaseB)} rows")

                 player_key season           team   age
2013      Diego López_ESP_A   1920       Espanyol  37.0
2695      Diego López_ESP_A   2122       Espanyol  39.0
3571      Diego López_ESP_B   2324       Valencia  21.0
3897      Diego López_ESP_B   2425       Valencia  22.0
2016       Javi López_ESP_A   1920       Espanyol  33.0
3274       Javi López_ESP_B   2324         Alavés  21.0
3867       Javi López_ESP_B   2425  Real Sociedad  22.0
5783  Ladislav Krejčí_CZE_A   1920        Bologna  27.0
3734  Ladislav Krejčí_CZE_B   2425         Girona  25.0
1915      Manu García_ESP_A   1920         Alavés  33.0
2242      Manu García_ESP_A   2021         Alavés  34.0
2578      Manu García_ESP_B   2122         Alavés  23.0



Re-saved 9483 rows


## Phase C — Eligible population, wide-format reshape, train/test split

In [32]:
import pandas as pd

df = pd.read_csv('../data/processed/player_seasons_merged.csv')

print("Shape:", df.shape)
print("\nseason dtype:", df['season'].dtype)
print("\nUnique season values:", sorted(df['season'].unique()))
print("\nplayer_key sample:", df['player_key'].sample(5, random_state=1).tolist())
print("\nDuplicate player_key+season rows:", df.duplicated(subset=['player_key', 'season']).sum())

Shape: (9483, 34)

season dtype: int64

Unique season values: [np.int64(1920), np.int64(2021), np.int64(2122), np.int64(2223), np.int64(2324), np.int64(2425)]

player_key sample: ['Gabriel Suazo_CHI', 'Moi Gómez_ESP', 'Iñigo Lekue_ESP', 'Mauro Arambarri_URU', 'Vito Mannone_ITA']

Duplicate player_key+season rows: 0


In [33]:
# Cast season to string so pandas treats it as a label, not a number
df['season'] = df['season'].astype(str)

# Verify the cast worked
print("New season dtype:", df['season'].dtype)
print("Unique season values:", sorted(df['season'].unique()))

# Re-save so this fix is actually persisted to disk, not just in memory
df.to_csv('../data/processed/player_seasons_merged.csv', index=False)
print("\nSaved. Re-loading fresh to confirm it stuck...")

# Paranoid check: reload from disk and confirm dtype survived the round-trip
check = pd.read_csv('../data/processed/player_seasons_merged.csv', dtype={'season': str})
print("Reloaded season dtype:", check['season'].dtype)
print("Reloaded shape:", check.shape)

New season dtype: str
Unique season values: ['1920', '2021', '2122', '2223', '2324', '2425']



Saved. Re-loading fresh to confirm it stuck...
Reloaded season dtype: str
Reloaded shape: (9483, 34)


In [34]:
# How many seasons does each player_key appear in?
seasons_per_player = df.groupby('player_key')['season'].nunique()

print("Distribution of seasons-per-player:")
print(seasons_per_player.value_counts().sort_index())

print("\nTotal unique players:", df['player_key'].nunique())

# Specifically: how many players HAVE a 2024-25 row at all?
# (if a player has no 2024-25 row, there's no target for them — they're not usable for training/eval)
has_2425 = df[df['season'] == '2425']['player_key'].unique()
print("\nPlayers with a 2024-25 row (potential targets):", len(has_2425))

Distribution of seasons-per-player:
season
1    1248
2     770
3     554
4     401
5     369
6     264
Name: count, dtype: int64

Total unique players: 3606

Players with a 2024-25 row (potential targets): 1557


In [35]:
# Restrict to players who actually have a 2024-25 target row
target_players = df[df['season'] == '2425']['player_key'].unique()

# For just those players, how many PRE-2024-25 seasons of history do they have?
history_df = df[(df['player_key'].isin(target_players)) & (df['season'] != '2425')]
history_counts = history_df.groupby('player_key')['season'].nunique()

# Players with a target but ZERO prior history (pure rookies in 2024-25)
zero_history = set(target_players) - set(history_counts.index)

print("Players with a 2024-25 target:", len(target_players))
print("\nDistribution of PRE-2024-25 history length, among players with a target:")
print(history_counts.value_counts().sort_index())
print("\nPlayers with a target but ZERO prior seasons (true rookies, no history to forecast from):", len(zero_history))

Players with a 2024-25 target: 1557

Distribution of PRE-2024-25 history length, among players with a target:
season
1    284
2    215
3    199
4    245
5    264
Name: count, dtype: int64

Players with a target but ZERO prior seasons (true rookies, no history to forecast from): 350


In [36]:
import pandas as pd

# Rebuild the eligible list (same logic as before, saved out this time)
required_seasons = {'1920', '2021', '2122', '2223', '2324'}
player_season_sets = df.groupby('player_key')['season'].apply(set)
has_all_prior = player_season_sets[player_season_sets.apply(lambda s: required_seasons.issubset(s))]
eligible_players = sorted([p for p in has_all_prior.index if '2425' in player_season_sets[p]])

# Save as a simple CSV: one player_key per row
eligible_df = pd.DataFrame({'player_key': eligible_players})
eligible_df.to_csv('../data/processed/phase_c_eligible_players.csv', index=False)

print("Total eligible players:", len(eligible_df))
print("\nFirst 10:")
print(eligible_df.head(10))
print("\nSaved to phase_c_eligible_players.csv")

Total eligible players: 264

First 10:
               player_key
0   Aaron Wan-Bissaka_COD
1        Aarón Martín_ESP
2  Abdoulaye Doucouré_MLI
3       Achraf Hakimi_MAR
4        Adam Marušić_MNE
5       Adrien Rabiot_FRA
6    Adrien Thomasson_FRA
7         Aihen Muñoz_ESP
8       Alassane Pléa_FRA
9        Alban Lafont_CIV

Saved to phase_c_eligible_players.csv


In [37]:
# Only the 264 eligible players, only their PRIOR seasons (never touch 2425 here)
feature_seasons = ['1920', '2021', '2122', '2223', '2324']
features_long = df[(df['player_key'].isin(eligible_players)) & (df['season'].isin(feature_seasons))].copy()

print("Rows going into the feature pivot:", len(features_long))
print("Expected: 264 players x 5 seasons =", 264 * 5)

# Sanity check: every eligible player should have EXACTLY 5 rows here, no more, no less
counts = features_long.groupby('player_key').size()
print("\nPlayers with exactly 5 rows:", (counts == 5).sum())
print("Players with != 5 rows (should be 0):", (counts != 5).sum())

Rows going into the feature pivot: 1320
Expected: 264 players x 5 seasons = 1320

Players with exactly 5 rows: 264
Players with != 5 rows (should be 0): 0


In [38]:
# --- Part A: numeric stats, pivoted across all 5 seasons ---

numeric_stats = ['matches', 'starts', 'minutes', 'goals', 'assists', 'goal_contributions',
                  'goals_excl_pk', 'pk_scored', 'pk_attempted', 'yellow_cards', 'red_cards',
                  'goals_p90', 'assists_p90', 'goal_contributions_p90', 'goals_excl_pk_p90',
                  'goal_contributions_excl_pk_p90', 'shots', 'shots_on_target', 'shot_accuracy',
                  'shots_p90', 'shots_on_target_p90', 'goals_per_shot', 'goals_per_shot_on_target',
                  'age']

wide_numeric = features_long.pivot(index='player_key', columns='season', values=numeric_stats)

# pivot() with multiple value columns creates a 2-level column structure like ('goals', '1920')
# flatten it into single strings like 'goals_1920' so it's easier to work with
wide_numeric.columns = [f"{stat}_{season}" for stat, season in wide_numeric.columns]
wide_numeric = wide_numeric.reset_index()

print("Wide numeric shape:", wide_numeric.shape)
print("Expected rows: 264, expected cols: 1 (player_key) +", len(numeric_stats), "x 5 =", 1 + len(numeric_stats)*5)
print("\nSample columns:", wide_numeric.columns.tolist()[:10])

Wide numeric shape:

 (264, 121)
Expected rows: 264, expected cols: 1 (player_key) + 24 x 5 = 121

Sample columns: ['player_key', 'matches_1920', 'matches_2021', 'matches_2122', 'matches_2223', 'matches_2324', 'starts_1920', 'starts_2021', 'starts_2122', 'starts_2223']


In [39]:
# --- Part B: categorical context, ONLY the 2023-24 (most recent PRIOR) season ---

context_2324 = df[(df['player_key'].isin(eligible_players)) & (df['season'] == '2324')][
    ['player_key', 'team', 'league', 'pos']
].copy()

# Rename columns to make clear these are specifically 2023-24 snapshots, not general context
context_2324 = context_2324.rename(columns={
    'team': 'team_2324',
    'league': 'league_2324',
    'pos': 'pos_2324'
})

print("Context shape:", context_2324.shape)
print("Expected: 264 rows, 4 columns (player_key + 3 context cols)")

# Sanity check: exactly one row per player (no duplicates, no missing players)
print("\nUnique player_keys in context:", context_2324['player_key'].nunique())
print("Duplicate player_key rows (should be 0):", context_2324.duplicated(subset='player_key').sum())

context_2324.head()

Context shape: (264, 4)
Expected: 264 rows, 4 columns (player_key + 3 context cols)

Unique player_keys in context: 264
Duplicate player_key rows (should be 0): 0


,player_key,team_2324,league_2324,pos_2324
1257,Bukayo Saka_ENG,Arsenal,ENG-Premier League,FW
1262,Gabriel Magalhães_BRA,Arsenal,ENG-Premier League,DF
1266,Kai Havertz_GER,Arsenal,ENG-Premier League,"FW,MF"
1267,Leandro Trossard_BEL,Arsenal,ENG-Premier League,"FW,MF"
1271,William Saliba_FRA,Arsenal,ENG-Premier League,DF


In [40]:
# --- Part C: the target - goals_excl_pk_p90 from 2024-25, and NOTHING else from that season ---

target_2425 = df[(df['player_key'].isin(eligible_players)) & (df['season'] == '2425')][
    ['player_key', 'goals_excl_pk_p90']
].copy()

target_2425 = target_2425.rename(columns={'goals_excl_pk_p90': 'target'})

print("Target shape:", target_2425.shape)
print("Expected: 264 rows, 2 columns (player_key + target)")
print("Duplicate player_keys (should be 0):", target_2425.duplicated(subset='player_key').sum())

# --- Final merge: numeric features + context + target, all joined on player_key ---

wide_final = wide_numeric.merge(context_2324, on='player_key', how='inner')
wide_final = wide_final.merge(target_2425, on='player_key', how='inner')

print("\nFinal wide dataset shape:", wide_final.shape)
print("Expected rows: 264, expected cols: 121 + 3 (context) + 1 (target) =", 121 + 3 + 1)

# Sanity checks before we trust this
print("\nAny nulls anywhere in the final table?", wide_final.isnull().sum().sum())
print("Any duplicate player_keys?", wide_final.duplicated(subset='player_key').sum())
print("\ntarget stats:")
print(wide_final['target'].describe())

Target shape: (264, 2)
Expected: 264 rows, 2 columns (player_key + target)
Duplicate player_keys (should be 0): 0

Final wide dataset shape: (264, 125)
Expected rows: 264, expected cols: 121 + 3 (context) + 1 (target) = 125

Any nulls anywhere in the final table? 0
Any duplicate player_keys? 0

target stats:
count    264.000000
mean       0.124459
std        0.163969
min        0.000000
25%        0.000000
50%        0.060000
75%        0.180000
max        1.070000
Name: target, dtype: float64


In [41]:
import pandas as pd

# Simplify position to primary position (take the first tag, e.g. "FW,MF" -> "FW")
wide_final['primary_pos'] = wide_final['pos_2324'].str.split(',').str[0]

# Bin the target into tiers - using quantiles so each tier has a similar number of players
wide_final['target_tier'] = pd.qcut(wide_final['target'], q=3, labels=['low', 'medium', 'high'])

# Cross-tab: how many players fall into each position x tier combination?
crosstab = pd.crosstab(wide_final['primary_pos'], wide_final['target_tier'])
print(crosstab)
print("\nSmallest group size:", crosstab.values[crosstab.values > 0].min())
print("Any groups with 0 players?", (crosstab.values == 0).sum())

target_tier  low  medium  high
primary_pos                   
DF            53      46     8
FW             0       5    49
GK            22       0     0
MF            13      40    28

Smallest group size: 5
Any groups with 0 players? 3


In [42]:
from sklearn.model_selection import train_test_split

# Stratified split: 80% train, 20% test, keeping position mix consistent between them
train_df, test_df = train_test_split(
    wide_final,
    test_size=0.2,
    stratify=wide_final['primary_pos'],
    random_state=42  # fixed seed so this split is reproducible if we rerun it later
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Verify the position mix actually stayed proportional in both sets
print("\nPosition breakdown - TRAIN (%):")
print((train_df['primary_pos'].value_counts(normalize=True) * 100).round(1))

print("\nPosition breakdown - TEST (%):")
print((test_df['primary_pos'].value_counts(normalize=True) * 100).round(1))

print("\nPosition breakdown - FULL dataset (%), for comparison:")
print((wide_final['primary_pos'].value_counts(normalize=True) * 100).round(1))

Train shape: (211, 127)
Test shape: (53, 127)

Position breakdown - TRAIN (%):
primary_pos
DF    40.3
MF    30.8
FW    20.4
GK     8.5
Name: proportion, dtype: float64

Position breakdown - TEST (%):
primary_pos
DF    41.5
MF    30.2
FW    20.8
GK     7.5
Name: proportion, dtype: float64

Position breakdown - FULL dataset (%), for comparison:
primary_pos
DF    40.5
MF    30.7
FW    20.5
GK     8.3
Name: proportion, dtype: float64


**Locked decision:** `team_2324` is dropped rather than one-hot encoded — 74 categories
is too sparse for 211 training rows. `league_2324` and `primary_pos` are one-hot encoded.

In [43]:
# Columns to exclude from the model entirely
# player_key: identifier, not a feature
# team_2324: excluded per our decision (74 categories, too sparse for 211 rows)
# pos_2324: raw multi-tag version, we use primary_pos instead
# target: this IS what we're predicting, doesn't belong in X
# primary_pos, target_tier: target_tier is a leak (derived from target itself) —
#   primary_pos we DO want, but as an encoded feature, so we handle it separately below

cols_to_drop = ['player_key', 'team_2324', 'pos_2324', 'target', 'target_tier']

# Build X and y for train
X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df['target']

# Build X and y for test
X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df['target']

# One-hot encode league_2324 and primary_pos
# pd.get_dummies converts a category column into multiple 0/1 columns,
# one per category (e.g. league_2324 becomes league_2324_PremierLeague,
# league_2324_LaLiga, etc., each 1 if the player is in that league, else 0)
X_train = pd.get_dummies(X_train, columns=['league_2324', 'primary_pos'])
X_test = pd.get_dummies(X_test, columns=['league_2324', 'primary_pos'])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nX_train columns:", X_train.columns.tolist())

X_train shape: (211, 129)
X_test shape: (53, 129)

X_train columns: ['matches_1920', 'matches_2021', 'matches_2122', 'matches_2223', 'matches_2324', 'starts_1920', 'starts_2021', 'starts_2122', 'starts_2223', 'starts_2324', 'minutes_1920', 'minutes_2021', 'minutes_2122', 'minutes_2223', 'minutes_2324', 'goals_1920', 'goals_2021', 'goals_2122', 'goals_2223', 'goals_2324', 'assists_1920', 'assists_2021', 'assists_2122', 'assists_2223', 'assists_2324', 'goal_contributions_1920', 'goal_contributions_2021', 'goal_contributions_2122', 'goal_contributions_2223', 'goal_contributions_2324', 'goals_excl_pk_1920', 'goals_excl_pk_2021', 'goals_excl_pk_2122', 'goals_excl_pk_2223', 'goals_excl_pk_2324', 'pk_scored_1920', 'pk_scored_2021', 'pk_scored_2122', 'pk_scored_2223', 'pk_scored_2324', 'pk_attempted_1920', 'pk_attempted_2021', 'pk_attempted_2122', 'pk_attempted_2223', 'pk_attempted_2324', 'yellow_cards_1920', 'yellow_cards_2021', 'yellow_cards_2122', 'yellow_cards_2223', 'yellow_cards_2324', '

In [44]:
# Confirm X_train and X_test have identical columns in identical order
print(list(X_train.columns) == list(X_test.columns))

True


`X_train`, `X_test`, `y_train`, `y_test` (129 features, log-space modeling target handled in 03) are now ready. Continue in `03_forecasting.ipynb`.